# Hospital Blood Pressure Data Cleaning & Transformation

## Overview
This dataset contains blood pressure readings for patients over multiple months. The data is currently in a **wide format** (each month as a separate column). Our goal is to:
- Clean and validate the data
- Transform from wide to long format using `melt()`
- Handle missing values and outliers
- Create meaningful visualizations
- Perform analysis on blood pressure trends

## Data Dictionary
- `Patient`: Unique patient identifier
- `Jan_BP`, `Feb_BP`, `Mar_BP`: Blood pressure readings (systolic) for January, February, March

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("HOSPITAL BLOOD PRESSURE DATA CLEANING")
print("="*70)

# Create the original dataframe
hospital = pd.DataFrame({
    "Patient": ["P001", "P002"],
    "Jan_BP": [120, 115],
    "Feb_BP": [118, 117],
    "Mar_BP": [116, 119]
})

print("\n[ORIGINAL DATA]")
print(f"Shape: {hospital.shape}")
print("\nFirst few rows:")
print(hospital)
print("\nData types:")
print(hospital.dtypes)

## Step 1: Data Validation & Quality Checks

Before transformation, we need to validate the data:
- Check for missing values
- Verify blood pressure ranges are medically valid (normal systolic: 90-140, but we'll use 80-200 as acceptable range)
- Check for duplicate patient records

In [ ]:
# Step 1: Data Validation
print("\n" + "="*70)
print("STEP 1: DATA VALIDATION")
print("="*70)

# Check for missing values
print("\n1. Missing Values Check:")
missing = hospital.isnull().sum()
print(f"Total missing values: {missing.sum()}")
if missing.sum() > 0:
    print("Missing values by column:")
    print(missing[missing > 0])
else:
    print("✓ No missing values found!")

# Check for duplicate patients
print("\n2. Duplicate Patient Check:")
duplicates = hospital['Patient'].duplicated().sum()
if duplicates > 0:
    print(f"⚠ Found {duplicates} duplicate patient records")
else:
    print("✓ No duplicate patient records found!")

# Check blood pressure ranges
print("\n3. Blood Pressure Range Validation:")
bp_columns = ['Jan_BP', 'Feb_BP', 'Mar_BP']
bp_values = hospital[bp_columns].values.flatten()

# Medical guidelines: Normal systolic BP is 90-140, but we use 80-200 as acceptable
outliers = bp_values[(bp_values < 80) | (bp_values > 200)]
if len(outliers) > 0:
    print(f"⚠ Found {len(outliers)} outlier BP readings: {outliers.tolist()}")
else:
    print("✓ All BP readings are within normal range (80-200)")

# Summary statistics
print("\n4. Summary Statistics for BP Readings:")
print(hospital[bp_columns].describe())

## Step 2: Transform Data from Wide to Long Format using `melt()`

The current **wide format** (months as columns) is not ideal for time-series analysis. We'll use `melt()` to convert to **long format** where each row represents one patient's reading for a specific month.

**Before melt (wide format):**
- Each patient has one row with multiple month columns

**After melt (long format):**
- Each patient-month combination has its own row
- Better for plotting trends, grouping, and statistical analysis

In [ ]:
# Step 2: Melt the dataframe from wide to long format
print("\n" + "="*70)
print("STEP 2: TRANSFORMING DATA USING melt()")
print("="*70)

# Original shape
print(f"\nOriginal shape (wide format): {hospital.shape}")
print("Original columns:", list(hospital.columns))

# Using melt to transform
# - id_vars: columns to keep as identifiers (Patient)
# - var_name: name for the new column that will hold the original column names (Month)
# - value_name: name for the new column that will hold the values (BP_Reading)
hospital_long = hospital.melt(
    id_vars=['Patient'],
    var_name='Month',
    value_name='BP_Reading'
)

# Display results
print(f"\nNew shape (long format): {hospital_long.shape}")
print("New columns:", list(hospital_long.columns))
print("\nTransformed data:")
print(hospital_long)

# Explanation of the transformation
print("\n[TRANSFORMATION EXPLANATION]")
print("✓ Each original row (patient) has been expanded into 3 rows (one per month)")
print("✓ The month names have been stored in the 'Month' column")
print("✓ The BP readings have been stored in the 'BP_Reading' column")

## Step 3: Clean and Enhance the Month Column

The month column currently contains values like 'Jan_BP', 'Feb_BP', etc. We'll:
- Extract just the month name
- Convert to proper month format for time-series analysis
- Add a numeric month column for ordering

In [ ]:
# Step 3: Clean and enhance the Month column
print("\n" + "="*70)
print("STEP 3: CLEANING MONTH COLUMN")
print("="*70)

# Create a copy to avoid modifying original
hospital_clean = hospital_long.copy()

# Method 1: Extract month name by removing '_BP' suffix
hospital_clean['Month'] = hospital_clean['Month'].str.replace('_BP', '')
print("\n1. Cleaned month names:")
print(hospital_clean['Month'].unique())

# Method 2: Add numeric month value for proper ordering (Jan=1, Feb=2, Mar=3)
month_mapping = {
    'Jan': 1,
    'Feb': 2,
    'Mar': 3,
    'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9,
    'Oct': 10, 'Nov': 11, 'Dec': 12
}

hospital_clean['Month_Num'] = hospital_clean['Month'].map(month_mapping)
print("\n2. Added numeric month column for ordering:")
print(hospital_clean[['Month', 'Month_Num']].drop_duplicates().sort_values('Month_Num'))

# Add full month name for better visualization
full_month_mapping = {'Jan': 'January', 'Feb': 'February', 'Mar': 'March'}
hospital_clean['Month_Full'] = hospital_clean['Month'].map(full_month_mapping)

# Add a formatted date column (assuming year 2024 for this example)
hospital_clean['Date'] = pd.to_datetime('2024-' + hospital_clean['Month_Num'].astype(str) + '-01')

print("\n3. Enhanced dataset with date information:")
print(hospital_clean)

# Verify data types
print("\n4. Data types after cleaning:")
print(hospital_clean.dtypes)

## Step 4: Blood Pressure Categorization

Based on medical guidelines, we'll categorize blood pressure readings into standard categories:
- **Low**: < 90 mmHg (Hypotension)
- **Normal**: 90-120 mmHg (Optimal)
- **Elevated**: 120-129 mmHg
- **High Stage 1**: 130-139 mmHg
- **High Stage 2**: 140-179 mmHg
- **Hypertensive Crisis**: ≥ 180 mmHg

In [ ]:
# Step 4: Categorize Blood Pressure Readings
print("\n" + "="*70)
print("STEP 4: BLOOD PRESSURE CATEGORIZATION")
print("="*70)

# Define a function to categorize BP based on medical guidelines
def categorize_bp(bp_value):
    """
    Categorize blood pressure reading based on standard medical guidelines
    
    Parameters:
    bp_value (int/float): Systolic blood pressure reading
    
    Returns:
    str: BP category
    """
    if pd.isna(bp_value):
        return 'Unknown'
    elif bp_value < 90:
        return 'Low (Hypotension)'
    elif 90 <= bp_value < 120:
        return 'Normal'
    elif 120 <= bp_value < 130:
        return 'Elevated'
    elif 130 <= bp_value < 140:
        return 'High Stage 1'
    elif 140 <= bp_value < 180:
        return 'High Stage 2'
    else:
        return 'Hypertensive Crisis'

# Apply categorization
hospital_clean['BP_Category'] = hospital_clean['BP_Reading'].apply(categorize_bp)

# Display results
print("\nData with BP Categories:")
print(hospital_clean[['Patient', 'Month', 'BP_Reading', 'BP_Category']])

# Show category distribution
print("\nBP Category Distribution:")
category_counts = hospital_clean['BP_Category'].value_counts()
for category, count in category_counts.items():
    percentage = (count / len(hospital_clean)) * 100
    print(f"  {category}: {count} ({percentage:.1f}%)")

# Add risk level indicator
risk_mapping = {
    'Low (Hypotension)': 'Moderate Risk',
    'Normal': 'Low Risk',
    'Elevated': 'Moderate Risk',
    'High Stage 1': 'High Risk',
    'High Stage 2': 'High Risk',
    'Hypertensive Crisis': 'Critical Risk'
}
hospital_clean['Risk_Level'] = hospital_clean['BP_Category'].map(risk_mapping)

## Step 5: Calculate Patient Statistics

For each patient, we'll calculate:
- Average blood pressure over the 3 months
- Minimum and maximum readings
- Trend (whether BP is improving, worsening, or stable)
- Variability (standard deviation)

In [ ]:
# Step 5: Patient-level statistics
print("\n" + "="*70)
print("STEP 5: PATIENT STATISTICS & TRENDS")
print("="*70)

# Group by patient and calculate statistics
patient_stats = hospital_clean.groupby('Patient').agg({
    'BP_Reading': ['mean', 'min', 'max', 'std', 'count']
}).round(1)

# Flatten column names
patient_stats.columns = ['BP_Avg', 'BP_Min', 'BP_Max', 'BP_Std', 'Readings_Count']
patient_stats = patient_stats.reset_index()

print("\nPatient Summary Statistics:")
print(patient_stats)

# Calculate trend (improving if BP is decreasing over time)
# Pivot to get BP values in order
bp_trend = hospital_clean.pivot(index='Patient', columns='Month_Num', values='BP_Reading')
# Sort columns by month number
bp_trend = bp_trend.reindex(sorted(bp_trend.columns), axis=1)

# Calculate trend direction
trend_results = []
for patient in bp_trend.index:
    values = bp_trend.loc[patient].dropna().values
    if len(values) >= 2:
        # Calculate slope (change over time)
        slope = values[-1] - values[0]
        if slope < -2:
            trend = 'Improving (↓)'
        elif slope > 2:
            trend = 'Worsening (↑)'
        else:
            trend = 'Stable'
    else:
        trend = 'Insufficient Data'
    trend_results.append({'Patient': patient, 'Trend': trend})

trend_df = pd.DataFrame(trend_results)
patient_stats = patient_stats.merge(trend_df, on='Patient')

print("\nPatient Statistics with Trends:")
print(patient_stats)

# Add average BP category for each patient
def get_avg_category(avg_bp):
    if avg_bp < 90:
        return 'Low'
    elif avg_bp < 120:
        return 'Normal'
    elif avg_bp < 130:
        return 'Elevated'
    elif avg_bp < 140:
        return 'High Stage 1'
    elif avg_bp < 180:
        return 'High Stage 2'
    else:
        return 'Crisis'

patient_stats['Avg_Category'] = patient_stats['BP_Avg'].apply(get_avg_category)
print("\nPatient Summary with Categories:")
print(patient_stats[['Patient', 'BP_Avg', 'Avg_Category', 'Trend']])

## Step 6: Visualizations

Creating meaningful visualizations to understand blood pressure trends and patterns:

In [ ]:
# Step 6: Visualizations
print("\n" + "="*70)
print("STEP 6: DATA VISUALIZATIONS")
print("="*70)

# Create a figure with multiple subplots
fig = plt.figure(figsize=(16, 12))

# Plot 1: Line plot showing BP trends over time for each patient
ax1 = fig.add_subplot(2, 3, 1)
for patient in hospital_clean['Patient'].unique():
    patient_data = hospital_clean[hospital_clean['Patient'] == patient]
    ax1.plot(patient_data['Month_Full'], patient_data['BP_Reading'], 
             marker='o', linewidth=2, markersize=8, label=patient)
ax1.set_xlabel('Month', fontsize=11, fontweight='bold')
ax1.set_ylabel('Blood Pressure (mmHg)', fontsize=11, fontweight='bold')
ax1.set_title('Blood Pressure Trends by Patient', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
# Add reference lines for BP categories
ax1.axhline(y=120, color='green', linestyle='--', alpha=0.5, label='Normal Upper Limit (120)')
ax1.axhline(y=140, color='orange', linestyle='--', alpha=0.5, label='High Stage 1 (140)')
ax1.axhline(y=90, color='blue', linestyle='--', alpha=0.5, label='Low BP (90)')

# Plot 2: Bar plot comparing average BP by patient
ax2 = fig.add_subplot(2, 3, 2)
avg_bp = hospital_clean.groupby('Patient')['BP_Reading'].mean().sort_values()
colors = ['#2ecc71' if x < 120 else '#e74c3c' if x > 140 else '#f39c12' for x in avg_bp.values]
bars = ax2.bar(avg_bp.index, avg_bp.values, color=colors, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Patient', fontsize=11, fontweight='bold')
ax2.set_ylabel('Average BP (mmHg)', fontsize=11, fontweight='bold')
ax2.set_title('Average Blood Pressure by Patient', fontsize=13, fontweight='bold')
# Add value labels on bars
for bar, value in zip(bars, avg_bp.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{value:.0f}', ha='center', va='bottom', fontweight='bold')
ax2.axhline(y=120, color='green', linestyle='--', alpha=0.5)
ax2.axhline(y=140, color='orange', linestyle='--', alpha=0.5)

# Plot 3: Monthly average BP (aggregated)
ax3 = fig.add_subplot(2, 3, 3)
monthly_avg = hospital_clean.groupby('Month_Full')['BP_Reading'].mean().reset_index()
# Ensure correct month order
month_order = ['January', 'February', 'March']
monthly_avg['Month_Full'] = pd.Categorical(monthly_avg['Month_Full'], categories=month_order, ordered=True)
monthly_avg = monthly_avg.sort_values('Month_Full')
ax3.plot(monthly_avg['Month_Full'], monthly_avg['BP_Reading'], 
         marker='s', linewidth=2, markersize=8, color='purple')
ax3.fill_between(monthly_avg['Month_Full'], monthly_avg['BP_Reading'], 
                  alpha=0.2, color='purple')
ax3.set_xlabel('Month', fontsize=11, fontweight='bold')
ax3.set_ylabel('Average BP (mmHg)', fontsize=11, fontweight='bold')
ax3.set_title('Monthly Average Blood Pressure (All Patients)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Distribution of BP readings
ax4 = fig.add_subplot(2, 3, 4)
sns.histplot(hospital_clean['BP_Reading'], bins=10, kde=True, color='teal', ax=ax4, alpha=0.7)
ax4.set_xlabel('Blood Pressure (mmHg)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax4.set_title('Distribution of BP Readings', fontsize=13, fontweight='bold')
ax4.axvline(x=120, color='green', linestyle='--', alpha=0.5, label='Normal Limit')
ax4.axvline(x=140, color='orange', linestyle='--', alpha=0.5, label='High Stage 1')
ax4.legend()

# Plot 5: Box plot comparison between patients
ax5 = fig.add_subplot(2, 3, 5)
sns.boxplot(x='Patient', y='BP_Reading', data=hospital_clean, palette='Set2', ax=ax5)
ax5.set_xlabel('Patient', fontsize=11, fontweight='bold')
ax5.set_ylabel('Blood Pressure (mmHg)', fontsize=11, fontweight='bold')
ax5.set_title('BP Distribution by Patient', fontsize=13, fontweight='bold')
# Add individual points to show all readings
sns.stripplot(x='Patient', y='BP_Reading', data=hospital_clean, color='black', alpha=0.5, size=8, ax=ax5)

# Plot 6: BP Category Pie Chart
ax6 = fig.add_subplot(2, 3, 6)
category_counts = hospital_clean['BP_Category'].value_counts()
colors_pie = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
wedges, texts, autotexts = ax6.pie(category_counts.values, 
                                      labels=category_counts.index,
                                      autopct='%1.1f%%',
                                      colors=colors_pie[:len(category_counts)],
                                      explode=[0.05] * len(category_counts))
ax6.set_title('Blood Pressure Category Distribution', fontsize=13, fontweight='bold')
# Style the percentage text
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

plt.tight_layout()
plt.show()

print("\n✓ All visualizations generated successfully!")

## Step 7: Final Data Quality Report

Summarize all transformations and provide a final clean dataset ready for analysis.

In [ ]:
# Step 7: Final Data Quality Report
print("\n" + "="*70)
print("FINAL DATA QUALITY REPORT")
print("="*70)

print("\n[TRANSFORMATION SUMMARY]")
print("-" * 50)
print(f"✓ Wide to Long transformation: {hospital.shape} → {hospital_clean.shape}")
print(f"✓ Total patients: {hospital_clean['Patient'].nunique()}")
print(f"✓ Total BP readings: {len(hospital_clean)}")
print(f"✓ Time period: {hospital_clean['Month_Full'].min()} to {hospital_clean['Month_Full'].max()}")
print(f"✓ BP range: {hospital_clean['BP_Reading'].min():.0f} - {hospital_clean['BP_Reading'].max():.0f} mmHg")
print(f"✓ Overall average BP: {hospital_clean['BP_Reading'].mean():.1f} mmHg")

print("\n[FINAL CLEAN DATAFRAME - First 6 rows]")
print("-" * 50)
print(hospital_clean.head(6).to_string())

print("\n[FINAL DATAFRAME INFO]")
print("-" * 50)
print(f"Shape: {hospital_clean.shape}")
print(f"\nColumns: {list(hospital_clean.columns)}")
print(f"\nData types:\n{hospital_clean.dtypes}")
print(f"\nMissing values:\n{hospital_clean.isnull().sum()}")

print("\n[DATA QUALITY CHECKS - PASSED]")
print("-" * 50)
checks = [
    ("No missing values in critical columns", 
     hospital_clean[['Patient', 'Month', 'BP_Reading']].isnull().sum().sum() == 0),
    ("Valid BP readings (80-200 range)", 
     (hospital_clean['BP_Reading'] >= 80).all() and (hospital_clean['BP_Reading'] <= 200).all()),
    ("Valid month values", 
     hospital_clean['Month'].isin(['Jan', 'Feb', 'Mar']).all()),
    ("No duplicate patient-month combinations", 
     hospital_clean.duplicated(subset=['Patient', 'Month']).sum() == 0)
]

for check_name, result in checks:
    status = "✓ PASSED" if result else "✗ FAILED"
    print(f"  {status}: {check_name}")

print("\n" + "="*70)
print("CLEANING AND TRANSFORMATION COMPLETE!")
print("="*70)
print("\nThe dataset is now ready for:")
print("  • Time-series analysis")
print("  • Patient trend monitoring")
print("  • Statistical modeling")
print("  • Reporting and dashboards")

# Optional: Save the cleaned data
# hospital_clean.to_csv('hospital_bp_cleaned.csv', index=False)
# print("\n✓ Cleaned data saved to 'hospital_bp_cleaned.csv'")

## Summary of Operations Performed

### 1. **Data Validation**
   - Checked for missing values, duplicates, and outliers
   - Validated blood pressure ranges against medical standards

### 2. **Data Transformation**
   - Used `melt()` to convert from wide to long format
   - Transformed 2 rows × 4 columns → 6 rows × 3 columns

### 3. **Feature Engineering**
   - Extracted month names and added numeric month ordering
   - Created date column for time-series analysis
   - Categorized BP readings into medical risk levels
   - Calculated patient statistics (avg, min, max, trend)

### 4. **Analysis**
   - Identified trends (improving, stable, or worsening)
   - Calculated overall and per-patient statistics
   - Generated distribution and category analysis

### 5. **Visualization**
   - Created line plots for trends over time
   - Generated bar charts, histograms, box plots, and pie charts
   - Added reference lines for medical guidelines

### Key Insights from this Dataset:
- Both patients maintained BP readings within normal to elevated ranges
- Patient P001 showed gradual improvement (120 → 116)
- Patient P002 remained relatively stable with a slight increase (115 → 119)
- No readings fell into high-risk categories

### Next Steps for Analysis:
1. Add more patients for statistical significance
2. Collect more months of data to identify long-term trends
3. Include diastolic BP readings for complete cardiovascular assessment
4. Add patient demographic data for subgroup analysis